In [2]:
import os
import numpy as np
import rioxarray
import xarray as xr
from scipy import stats
from sklearn.model_selection import train_test_split

# Load the TIF files
sentinel_stack = rioxarray.open_rasterio('sentinel_image.tif')
labeled_map = rioxarray.open_rasterio('labeled_data.tif').squeeze()

print(f"Sentinel Shape: {sentinel_stack.shape}")
print(f"Labels Shape: {labeled_map.shape}")

Sentinel Shape: (9, 9336, 5763)
Labels Shape: (9336, 5763)


In [ ]:
# Re-assign band names 
band_names = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B11', 'B12']
sentinel_stack.coords['band'] = band_names

# Select the 6 Master Bands for project (Red, Green, Blue, NIR, SWIR1, SWIR2)
s2_6band = sentinel_stack.sel(band=['B04', 'B03', 'B02', 'B08', 'B11', 'B12'])

# Normalize to 0-1 range and Clip
s2_norm = (s2_6band / 10000.0).clip(0, 1)

print("Normalization complete. Values now range from 0.0 to 1.0.")
print(f"Selected Bands: {s2_norm.band.values}")

Normalization complete. Values now range from 0.0 to 1.0.
Selected Bands: ['B04' 'B03' 'B02' 'B08' 'B11' 'B12']


In [ ]:
import numpy as np
from scipy import stats

patch_size = 256
stride = 256 

X_patches = []
Y_masks = []
Y_labels = []

# Balancing parameters
WATER_ID = 80
# Keeping 45% of 304 water patches (reduce class imbalance)
WATER_KEEP_PROBABILITY = 0.45 


_, height, width = s2_norm.shape 

print("Starting Balanced Tiling with NDVI...")

for y in range(0, height - patch_size + 1, stride): 
    for x in range(0, width - patch_size + 1, stride): 
        
        # Extract spectral image patch
        img_patch = s2_norm.isel(y=slice(y, y + patch_size), x=slice(x, x + patch_size)).values
        img_patch = np.transpose(img_patch, (1, 2, 0))
        
        # Extract corresponding label mask patch
        mask_patch = labeled_map.isel(y=slice(y, y + patch_size), x=slice(x, x + patch_size)).values
        
        # Skip if "No Data" pixels exceed 15% of the patch
        if np.mean(mask_patch == 0) > 0.15: 
            continue
        
        # Determine the Patch-Level Label (Mode)
        mode_res = stats.mode(mask_patch, axis=None, keepdims=True)
        patch_label = int(np.ravel(mode_res.mode)[0])
        
        # Probabilistic skip for Water
        if patch_label == WATER_ID:
            if np.random.random() > WATER_KEEP_PROBABILITY:
                continue 
        
        # Calculate NDVI (Red is Index 0, NIR is Index 3)
        red_chan = img_patch[:, :, 0]
        nir_chan = img_patch[:, :, 3]
        
        ndvi = (nir_chan - red_chan) / (nir_chan + red_chan + 1e-10)
        ndvi = ndvi[:, :, np.newaxis]
        
        # Stack NDVI onto the 6-band image
        img_patch_7b = np.concatenate([img_patch, ndvi], axis=-1)

        # Save data
        X_patches.append(img_patch_7b)
        Y_masks.append(mask_patch)
        Y_labels.append(patch_label)

# Convert to final numpy arrays
X_patches = np.array(X_patches)
Y_masks = np.array(Y_masks)
Y_labels = np.array(Y_labels)

print(f"Tiling Complete!")
print(f"Total patches: {len(X_patches)}")
print(f"Input Shape: {X_patches.shape}")

Starting Balanced Tiling with NDVI...
Tiling Complete!
Total patches: 625
Input Shape: (625, 256, 256, 7)


In [5]:
# Splitting into 80% Training and 20% Testing
X_train, X_test, Y_train_mask, Y_test_mask, Y_train_lab, Y_test_lab = train_test_split(
    X_patches, Y_masks, Y_labels, 
    test_size=0.20, 
    random_state=30, 
    stratify=Y_labels
)

print(f"Training set: {len(X_train)} patches")
print(f"Testing set: {len(X_test)} patches")

Training set: 500 patches
Testing set: 125 patches


In [ ]:
# Offline Data Augmentation for Tree Cover & Shrubland
X_train_aug = []
Y_train_lab_aug = []
Y_train_mask_aug = []

for i in range(len(X_train)):
    img, mask, label = X_train[i], Y_train_mask[i], Y_train_lab[i]
    
    X_train_aug.append(img)
    Y_train_lab_aug.append(label)
    Y_train_mask_aug.append(mask)

    if label == 10: # Tree cover (22 patches -> 110 patches)
        # four variations: 90, 180, 270 rotations + 1 horizontal flip
        for k in [1, 2, 3]: 
            X_train_aug.append(np.rot90(img, k))
            Y_train_lab_aug.append(label)
            Y_train_mask_aug.append(np.rot90(mask, k))
        
        X_train_aug.append(np.fliplr(img))
        Y_train_lab_aug.append(label)
        Y_train_mask_aug.append(np.fliplr(mask))
        
    elif label == 20: # SHRUBLAND (51 patches -> 102 patches)
        # one variation: Horizontal Flip
        X_train_aug.append(np.fliplr(img))
        Y_train_lab_aug.append(label)
        Y_train_mask_aug.append(np.fliplr(mask))

# Convert back to arrays
X_train = np.array(X_train_aug)
Y_train_lab = np.array(Y_train_lab_aug)
Y_train_mask = np.array(Y_train_mask_aug)

print(f"Final Augmented Training Set: {len(X_train)} patches")

Final Augmented Training Set: 639 patches


In [ ]:
import collections
import numpy as np

def print_detailed_stats(X, Y_lab, Y_mask, set_name, class_map):
    print(f"\n{'='*20} {set_name.upper()} SET STATS {'='*20}")
    print(f"Total Patches: {len(X)}")
    
    # Patch-level Distribution (Labels)
    label_counts = collections.Counter(Y_lab)
    print("\n--- PATCH-LEVEL (Classification) ---")
    for class_id in sorted(class_map.keys()):
        if class_id in label_counts:
            count = label_counts[class_id]
            percentage = (count / len(Y_lab)) * 100
            name = class_map[class_id][0]
            print(f"Class {class_id:2} ({name:15}): {count:4} patches ({percentage:5.1f}%)")

    # Pixel-level Distribution (Masks)
    unique_px, counts_px = np.unique(Y_mask, return_counts=True)
    pixel_counts = dict(zip(unique_px, counts_px))
    total_pixels = Y_mask.size
    
    print("\n--- PIXEL-LEVEL (Segmentation) ---")
    for class_id in sorted(class_map.keys()):
        if class_id in pixel_counts:
            px_count = pixel_counts[class_id]
            px_perc = (px_count / total_pixels) * 100
            name = class_map[class_id][0]
            print(f"Class {class_id:2} ({name:15}): {px_count:10} px ({px_perc:5.1f}%)")
    
    if 0 in pixel_counts:
        print(f"\n Note: Found {pixel_counts[0]} 'No Data' (Class 0) pixels.")

class_map = {
    10: ("Tree cover", "#006400"),
    20: ("Shrubland", "#ffbb22"),
    30: ("Grassland", "#ffff4c"),
    40: ("Cropland", "#f096ff"),
    50: ("Built-up", "#fa0000"),
    80: ("Permanent water", "#0064ff"),
}

print_detailed_stats(X_train, Y_train_lab, Y_train_mask, "Training (Augmented)", class_map)
print_detailed_stats(X_test, Y_test_lab, Y_test_mask, "Testing (Original)", class_map)


==================== TRAINING (AUGMENTED) SET STATS ====================
Total Patches: 639

--- PATCH-LEVEL (Classification) ---
Class 10 (Tree cover     ):  110 patches ( 17.2%)
Class 20 (Shrubland      ):  102 patches ( 16.0%)
Class 30 (Grassland      ):  117 patches ( 18.3%)
Class 40 (Cropland       ):  112 patches ( 17.5%)
Class 50 (Built-up       ):   88 patches ( 13.8%)
Class 80 (Permanent water):  110 patches ( 17.2%)

--- PIXEL-LEVEL (Segmentation) ---
Class 10 (Tree cover     ):    6236627 px ( 14.9%)
Class 20 (Shrubland      ):    6704273 px ( 16.0%)
Class 30 (Grassland      ):    9238845 px ( 22.1%)
Class 40 (Cropland       ):    6750567 px ( 16.1%)
Class 50 (Built-up       ):    5275288 px ( 12.6%)
Class 80 (Permanent water):    7254461 px ( 17.3%)

==================== TESTING (ORIGINAL) SET STATS ====================
Total Patches: 125

--- PATCH-LEVEL (Classification) ---
Class 10 (Tree cover     ):    6 patches (  4.8%)
Class 20 (Shrubland      ):   13 patches ( 10.4%

In [8]:
save_dir = "processed_data"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# Save as .npy files 
np.save(os.path.join(save_dir, 'X_train.npy'), X_train)
np.save(os.path.join(save_dir, 'X_test.npy'), X_test)
np.save(os.path.join(save_dir, 'Y_train_mask.npy'), Y_train_mask)
np.save(os.path.join(save_dir, 'Y_test_mask.npy'), Y_test_mask)
np.save(os.path.join(save_dir, 'Y_train_labels.npy'), Y_train_lab)
np.save(os.path.join(save_dir, 'Y_test_labels.npy'), Y_test_lab)

print(f"All datasets saved successfully in '{save_dir}' folder.")

All datasets saved successfully in 'processed_data' folder.
